In [1]:
!pip install langchain langchain-huggingface huggingface-hub  duckduckgo-search

   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   ---------------------------------------- 3.9/3.9 MB 45.7 MB/s  0:00:00

   ----- ----------------------------------  2/15 [primp]
   ------------- --------------------------  5/15 [requests-toolbelt]
   ------------- --------------------------  5/15 [requests-toolbelt]
   ---------------- -----------------------  6/15 [duckduckgo-search]
   ------------------ ---------------------  7/15 [langsmith]
   ------------------ ---------------------  7/15 [langsmith]
   ------------------ ---------------------  7/15 [langsmith]
   ------------------ ---------------------  7/15 [langsmith]
   ------------------ ---------------------  7/15 [langsmith]
   --------------------- ------------------  8/15 [langgraph-sdk]
   --------------------- ------------------  8/15 [langgraph-sdk]
   --------------------- ------------------  8/15 [langgraph-sdk]
   --------------------- ------------------  8/15 [langgraph-sdk]
   -----------

In [2]:
!pip install html2text faiss-cpu streamlit chromadb langchain-community

   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 9.1/9.1 MB 46.8 MB/s  0:00:00
   ---------------------------------------- 0.0/797.0 kB ? eta -:--:--
   ---------------------------------------- 797.0/797.0 kB 33.4 MB/s  0:00:00
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------  11.3/11.3 MB 58.9 MB/s eta 0:00:01
   ---------------------------------------- 11.3/11.3 MB 54.4 MB/s  0:00:00
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   -------------------------------- ------- 12.8/15.8 MB 62.0 MB/s eta 0:00:01
   ---------------------------------------- 15.8/15.8 MB 55.3 MB/s  0:00:00
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 36.0 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
docling 2.88.0 requires typer<0.22.0,>=0.12.5, but you have typer 0.24.1 which is incompatible.
llama-index-embeddings-huggingface 0.2.3 requires llama-index-core<0.11.0,>=0.10.1, but you have llama-index-core 0.14.20 which is incompatible.
llama-index-indices-managed-llama-cloud 0.1.6 requires llama-index-core<0.11.0,>=0.10.0, but you have llama-index-core 0.14.20 which is incompatible.
llama-index-llms-groq 0.1.4 requires llama-index-core<0.11.0,>=0.10.1, but you have llama-index-core 0.14.20 which is incompatible.
llama-index-llms-openai-like 0.1.3 requires llama-index-core<0.11.0,>=0.10.1, but you have llama-index-core 0.14.20 which is incompatible.
llama-index-llms-openai-like 0.1.3 requires llama-index-llms-openai<0.2.0,>=0.1.1, but you have llama-index-llms-openai 0.7.5 which is incompatible.
llama-index-mu

In [3]:
!pip install numexpr youtube_search wikipedia

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11785 sha256=a82eee74ff315cf1782198facc6d792878064cf6c51e8c313c9178671aa810b1
  Stored in directory: c:\users\dorot\appdata\local\pip\cache\wheels\8f\ab\cb\45ccc40522d3a1c41e1d2ad53b8f33a62f394011ec38cd71c6
Successfully built wikipedia

   ------------- -------------------------- 1/3 [youtube_search]
   ---------------------------------------- 3/3 [wikipedia]



In [6]:
import pandas as pd
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

def measure_model_performance(model_name, max_length, input_text,
                             model_size_billion_params):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    
    input_tokens = tokenizer(input_text, return_tensors="pt")
    
    start_time = time.time()
    output = model.generate(**input_tokens, max_length=max_length)
    end_time = time.time()
    
    output_text = tokenizer.decode(output[0], skip_special_tokens=True)
    
    time_taken = end_time - start_time
    
    num_tokens = output.size(1)
    
    tokens_per_second = num_tokens / time_taken
    
    tokens_per_second_per_billion_params = tokens_per_second / model_size_billion_params
    
    data = {
        "model_name": model_name.split('/')[1],
        "billion_parameters": model_size_billion_params,
        "tokens_per_second": tokens_per_second,
        "tokens_per_second_per_billion_parameters": tokens_per_second_per_billion_params,
        "answer": output_text
    }
    
    return data



In [10]:
df = pd.DataFrame()

models = [
    "mistralai/Mistral-7B-Instruct-v0.1",
    "microsoft/Phi-3-mini-4k-instruct",
    "Qwen/Qwen2-7B-Instruct",
    "tri-ml/mamba-7b-rw",
    "meta-llama/Meta-Llama-3-8B-Instruct"
]

input_texts = [
    "[INST]describe briefly what is an AI agent[/INST]",
    "<|user|>describe briefly what is an AI agent<|end|><|assistant|>",
    "describe briefly what is an AI agent",
    "describe briefly what is an AI agent",
    "describe briefly what is an AI agent"
]

parameters = [7, 3, 7, 7, 8]

# Ensure both lists have the same length
assert len(models) == len(input_texts), "The number of models must match the number of input texts."

results = []
for model_name, input_text, model_size_billion_params in zip(models, input_texts, parameters):
    data = measure_model_performance(model_name, max_length=512, input_text=input_text, model_size_billion_params=model_size_billion_params)
    results.append(data)

df = pd.DataFrame(results)
df

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

: 

In [ ]:

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.bar(df['model_name'], df['tokens_per_second'], color='skyblue')
plt.xlabel('Model Name')
plt.ylabel('Tokens per second')
plt.title('Tokens per second by model')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('Token_second.jpg', format='jpeg')
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(df['model_name'], df['tokens_per_second_per_billion_parameters'], color='skyblue')
plt.xlabel('Model Name')
plt.ylabel('Tokens per second \n for billion parameters')
plt.title('Tokens per second for billion parameters')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('Token_second_B_param.jpg', format='jpeg')
plt.show()

In [ ]:
for i in range(5):
    print(df.loc[i,'model_name'])
    print(df.loc[i,'answer'])

In [ ]:
data = {
    "Model": [
        "Mistral-7B-Instruct-v0.1",
        "Phi-3-mini-4k-instruct",
        "Qwen2-7B-Instruct",
        "mamba-7b-rw",
        "Meta-Llama-3-8B-Instruct"
    ],
    "Overall Quality": [8, 9, 10, 1, 8],
    "Completeness": [8, 9, 10, 1, 8],
    "Truthfulness": [9, 9, 10, 1, 9]
}

df = pd.DataFrame(data)

df.set_index("Model").plot(kind="bar", figsize=(12, 6), color=['#1f77b4', '#ff7f0e', '#2ca02c'])
plt.title('Scores for AI models')
plt.xlabel('Models')
plt.ylabel('Scores')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Score Type')
plt.tight_layout()
plt.savefig('evaluation.jpg', format='jpeg')
plt.show()

In [ ]:
from langchain.tools import DuckDuckGoSearchRun

ddg_search = DuckDuckGoSearchRun()
ddg_search.run('Who is the current president of Italy?')

In [ ]:

from langchain.agents import Tool

tools = [
   Tool(
       name="DuckDuckGo Search",
       func=ddg_search.run,
       description="A web search tool to extract information from Internet.",
   )
]

In [ ]:
import os
SERPER_API_KEY = 'your_key'
os.environ["SERPER_API_KEY"] = SERPER_API_KEY

from langchain.utilities import GoogleSerperAPIWrapper

google_search = GoogleSerperAPIWrapper()

tools.append(
   Tool(
       name="Google Web Search",
       func=google_search.run,
       description="Google search tool to extract information from Internet.",
   )
)

In [ ]:
from langchain.tools import WikipediaQueryRun
from langchain.utilities import WikipediaAPIWrapper

wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

tools.append(
   Tool(
       name="Wikipedia Web Search",
       func=wikipedia.run,
       description="Useful tool to search Wikipedia.",
   )
)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain.agents import load_tools
from langchain.agents import initialize_agent
model_id = "microsoft/Phi-3-mini-4k-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    load_in_4bit=True,
)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=500, top_k=50, temperature=0.1)

llm = HuggingFacePipeline(pipeline=pipe)


agent = initialize_agent(
   tools, llm, agent="zero-shot-react-description", verbose=True,
    handle_parsing_errors=True,
    max_iterations=1,
)

query = "Who is the current president of Italy? Who was the previous one?"

response = agent.run(query)
print(response)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain.agents import load_tools, AgentExecutor, initialize_agent
from langchain_core.prompts import PromptTemplate

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    load_in_4bit=True,
)

pipe = pipeline("text-generation", model=model, 
                tokenizer=tokenizer, max_new_tokens=500, 
                top_k=50, temperature=0.1, 
               do_sample=True)

llm = HuggingFacePipeline(pipeline=pipe)

tools = load_tools(["ddg-search",  "llm-math", "wikipedia"], llm=llm)

template = '''Answer the following question as best as you can. You have access to the following tools:

{tools}

Use the following format:

Question: {input}
Thought: You should think about what action to take
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Do not answer or ask any other questions. Stop once you have provided the Final Answer.


Begin!

Question: {input}
'''

prompt = PromptTemplate.from_template(template)

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent="zero-shot-react-description",
    prompt=prompt,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=1,
    stop_sequence="Final Answer:" 
)

query = "The biography of Napoleon"

response = agent.run(query)
print(response)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain.agents import load_tools
from langchain.agents import initialize_agent
model_id = "mistralai/Mistral-7B-Instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    load_in_4bit=True,
)